# Transaction Foundation Model on Ray — Part 6: Downstream fraud — raw vs FM vs fusion

<div align="left">
  <a target="_blank" href="https://console.anyscale.com/template-preview/fintech_transaction_fm"><img src="https://img.shields.io/badge/🚀 Run_on-Anyscale-9hf"></a>&nbsp;
  <a href="https://github.com/anyscale/templates/tree/main/templates/fintech_transaction_fm" role="button"><img src="https://img.shields.io/static/v1?label=&message=View%20On%20GitHub&color=586069&logo=github&labelColor=2f363d"></a>&nbsp;
</div>

**⏱️ Time to complete**: ~5 min

---

This is the payoff of the whole series: does the foundation-model embedding actually help catch fraud, compared to the features a team already has? We answer it with a controlled experiment, and — unlike the earlier stages — this one isn't a Ray job. The distributed work already happened upstream; here we fit a single-node XGBoost on the embeddings Part 5 produced.

In [ ]:
import sys, os, json

DEMO_ROOT = os.path.abspath(os.getcwd())
if DEMO_ROOT not in sys.path:
    sys.path.insert(0, DEMO_ROOT)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.paths import artifact_paths, get_demo_base_dir

SCALE = "mini"                       # same knob as the earlier parts
paths = artifact_paths(get_demo_base_dir(), SCALE)

## Does the embedding beat the features you already have?

We train the **same XGBoost** on three feature sets, so the only thing that changes is the *representation*:

1. **raw** — the target transaction's own fields (amount, hour, day, MCC): the hand-built baseline a fraud team has today.
2. **fm** — only the FM embedding of the card's history window, with none of the raw fields.
3. **fusion** — the embedding concatenated with the raw fields (Nubank's joint-fusion recipe).

The lift of **fm** and **fusion** over **raw** is the case for a transaction foundation model: it says you can drop or augment a hand-tuned feature pipeline with one pretrained encoder.

The evaluation follows NVIDIA's transaction-FM blueprint so the numbers are comparable: the **temporal 80/10/10 split** from Part 2 (train on the past, test on the most recent 10% — no leakage), per-transaction last-event labels, and **PR-AUC at natural fraud prevalence** (the downsampled normals are importance-weighted back to ~0.1%). An eval fingerprint makes two runs comparable only if they scored the exact same held-out set.

In [ ]:
from src.downstream import run_downstream, print_summary

summary = run_downstream(paths["embeddings"], paths["downstream"])
print_summary(summary)

## Reading this honestly at `mini` scale

At `mini` the foundation model trailed the raw baseline — and that's the expected, honest result, not a bug. The encoder here was pretrained for **2 CPU epochs at 64 dimensions, 2 layers**: it has barely learned, so a 64-dim summary of recent history can't yet compete with the target transaction's own raw fields. So `mini` is a smoke test of the *pipeline and the evaluation harness*, not a model you'd ship.

What transfers regardless of scale is the **methodology**: a leakage-free temporal split, metrics reported at the true ~0.1% prevalence (where the published TFMs are measured), and a controlled comparison where representation is the only variable. Run the `small`/`full` GPU pretrain and the picture inverts — `fm` and `fusion` overtake `raw`, matching the lift NVIDIA's blueprint and FATA-Trans report on this dataset. The point of this notebook is the apparatus that will *show* that lift, run honestly at a scale that fits on a CPU.

## The curves

Two views of the same held-out scores, both at natural prevalence. **ROC** shows how well each representation *ranks* fraud above normal — readable here because the `mini` model is weak (at `full` a strong model pushes AUC-ROC toward 1.0, where it saturates and hides differences). **Precision–Recall** shows the operational reality: at ~0.1% fraud, precision collapses, which is exactly why PR-AUC — not accuracy or ROC — is the number a fraud team actually optimizes.

In [ ]:
from sklearn.metrics import roc_curve, precision_recall_curve, roc_auc_score, average_precision_score

pred = pd.read_parquet(os.path.join(paths["downstream"], "test_predictions.parquet"))
sns.set_theme(style="white", context="notebook")
COLORS = {"raw": "#4C78A8", "fm": "#F58518", "fusion": "#54A24B"}

fig, (ax_roc, ax_pr) = plt.subplots(1, 2, figsize=(12, 4.8))
for fs in ["raw", "fm", "fusion"]:
    d = pred[pred["feature_set"] == fs]
    fpr, tpr, _ = roc_curve(d["label"], d["proba"], sample_weight=d["weight"])
    auc = roc_auc_score(d["label"], d["proba"], sample_weight=d["weight"])
    ax_roc.plot(fpr, tpr, color=COLORS[fs], lw=1.8, label=f"{fs} (AUC {auc:.3f})")
    prec, rec, _ = precision_recall_curve(d["label"], d["proba"], sample_weight=d["weight"])
    ap = average_precision_score(d["label"], d["proba"], sample_weight=d["weight"])
    ax_pr.plot(rec, prec, color=COLORS[fs], lw=1.8, label=f"{fs} (PR-AUC {ap:.3f})")

ax_roc.plot([0, 1], [0, 1], ls="--", color="#cccccc", lw=1)
ax_roc.set_title("ROC — ranking fraud above normal")
ax_roc.set_xlabel("false positive rate"); ax_roc.set_ylabel("true positive rate"); ax_roc.legend()
ax_pr.set_title("Precision–Recall — at ~0.1% prevalence")
ax_pr.set_xlabel("recall"); ax_pr.set_ylabel("precision"); ax_pr.set_xlim(0, 1); ax_pr.legend()
for ax in (ax_roc, ax_pr):
    sns.despine(ax=ax)
plt.tight_layout()
plt.show()

## Takeaways

- **The comparison is controlled**: one XGBoost recipe, three feature sets, representation the only variable — so any lift is attributable to the embedding, not to tuning.
- **The protocol is honest and comparable**: temporal split (no leakage), per-transaction labels, PR-AUC at the true ~0.1% prevalence, and an eval fingerprint so two runs are only compared when they scored the same held-out set. It matches NVIDIA's transaction-FM blueprint.
- **At `mini` the FM trails raw** — a 2-epoch, 64-dim CPU encoder is undertrained. The value of this stage is the apparatus; the lift shows up with the `small`/`full` GPU pretrain, where pretrained TFMs beat hand-built features on this benchmark.
- **Not every stage is a Ray job.** The distributed work (tokenize, pretrain, embed) is done; scoring is a single-node fit. Reaching for the cluster here would add nothing — using the right tool per stage is part of the engineering.

---

## Next

**Part 7 — Online serving with Ray Serve**: deploy the encoder behind an HTTP endpoint that returns an embedding and a fraud score in one call, caching the static card-level fields and autoscaling replicas under load.